# $$\text{Exercise 7 - Simon M. Løvdal}$$

## $$\text{Note:}$$

```bash
# Change directory
cd $FOAM_RUN

# Copy Cavity
cp -r $FOAM_TUTORIALS/incompressible/icoFoam/cavity/cavity .

# Generate mesh
blockMesh

# Edit blockMesh
gedit system/blockMeshDict &

# Viewing results
paraFoam

# Removing results
foamListTimes -rm

# Add the streamfunction (post process)
icoFoam -postProcess -func streamFunction

# Removing Directory
rmdir directoryName
```

### $\text{Task a)}$

$$
\text{Remember}\quad \text{Re} = \frac{u \cdot L}{\nu}
$$
$$
\nu = 0.01,\ U = 1 \text{m}/\text{s},\ L = 1\text{m},\ \Delta t = 0.01
$$

We have to change $\nu = 0.01 \to 0.001$ for $\text{Re} = 1000$!

![description](U_magnitude_Re1000.png)

![description](psi_constant_Re1000.png)

```bash
# Printout for last calculation!
Time = 100

Courant Number mean: 0.226344 max: 0.608627
smoothSolver:  Solving for Ux, Initial residual = 8.55794e-08, Final residual = 8.55794e-08, No Iterations 0
smoothSolver:  Solving for Uy, Initial residual = 8.44877e-08, Final residual = 8.44877e-08, No Iterations 0
DICPCG:  Solving for p, Initial residual = 1.22748e-06, Final residual = 6.63334e-07, No Iterations 1
time step continuity errors : sum local = 7.01022e-09, global = -4.74974e-19, cumulative = 2.39356e-18
DICPCG:  Solving for p, Initial residual = 6.99878e-07, Final residual = 6.99878e-07, No Iterations 0
time step continuity errors : sum local = 7.25026e-09, global = -9.17866e-19, cumulative = 1.47569e-18
ExecutionTime = 0.61 s  ClockTime = 1 s

```

Note that as we increase the time step, even though the program does not crash, the courant number increases. For $\Delta \text{t} = 0.1$ we get a $\text{Courant number mean} \approx 4.4$. This leads to stability concerns, but since icoFoam uses implicit methods this is not causing the program to explode. Instead, the solution reduces in accuracy. Even though I run icoFoam using the same parameters, for $\Delta \text{t} = 0.1$ the solution seems to converge slower!

### $\text{Task b)}$

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>Gauss Linear</strong></p>
    <img src="psi_constant_Re1000.png" width="1200">
  </div>

  <div>
    <p><strong>Gauss Upwind Scheme</strong></p>
    <img src="psi_constant_upwind_scheme.png" width="1200">
  </div>
</div>

Even though we changed the scheme, we get sort of the same streamline distrubution. What's different is that the Upwind Scheme is of $\mathcal{O}(\Delta x)$ (first order), which introduces more diffusion compared to the Gauss Linear Scheme! Even for $\Delta \text{t} = 1$ we approximately the same results for the Gauss Upwind Scheme.

### $\text{Task c)}$

For the pitzDaily setup:

$$
\nu = 10^{-5},\ \text{Scale} = 0.001\text{m} = 1\text{mm},\ u = 10\text{m}/\text{s}
$$
$$
\text{Re} = \frac{u \cdot L}{\nu} = \frac{10 \cdot 0.001}{10^{-5}} = 1000
$$

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>turbulenceProperties: Turbulence on</strong></p>
    <img src="turbulence_nonlaminar.png" width="1200">
  </div>

  <div>
    <p><strong>turbulenceProperties: Turbulence off</strong></p>
    <img src="no_turbulence_nonlaminar.png" width="1200">
  </div>
</div>

Neither of these are laminar. There is clearly some turbulence occuring under the step! Here I just changed the setting in the turbulenceProperties directory.

```bash
RAS
{
    // Tested with kEpsilon, realizableKE, kOmega, kOmegaSST,
    // ShihQuadraticKE, LienCubicKE.
    RASModel        kEpsilon;

    turbulence      on; # <---- 

    printCoeffs     on;
}
```

Let's leave this on, and try to lower the Reynolds number instead!

Let's first try by increasin the scale in the blockMeshDict from $0.001 \to 0.1$. This did not work especially well.. Therefore let's lower $\nu = 10^{-5} \to 0.001$. Now the $\text{Re} = 10$.

#### $$\text{Comparison between Re=1000 v.s. Re=10}$$

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>Re = 1000</strong></p>
    <img src="turbulence_nonlaminar.png" width="1200">
  </div>

  <div>
    <p><strong>turbulenceProperties: Turbulence on & Re=10</strong></p>
    <img src="nonlaminar_Re10.png" width="1200">
  </div>
</div>

Reducing $\nu \to 0.001$ helped a little, but let's keep on reducing it $\nu \to 0.01$. Thus $\text{Re} = 1$:

#### $$\text{Comparison between Re=10 v.s. Re=1}$$

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>Re = 10</strong></p>
    <img src="nonlaminar_Re10.png" width="1200">
  </div>

  <div>
    <p><strong>Re=1</strong></p>
    <img src="laminar_Re1.png" width="1200">
  </div>
</div>

The flow is now definitely laminar.

### $\text{Task d)}$

```bash
blocks
(
    hex (0 1 2 3 4 5 6 7) (20 20 1) 
    simpleGrading #This is what changed inside blockMeshDict
	(
		(
		(0.5 0.5 4)
		(0.5 0.5 0.25)
		)
		(
		(0.5 0.5 4)
		(0.5 0.5 0.25)
		)
		1
	)
);

```

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>Simple Grading</strong></p>
    <img src="simpleGrading.png" width="1200">
  </div>

  <div>
    <p><strong>Mesh stretching</strong></p>
    <img src="simleGrading_stretching2.png" width="1200">
  </div>
</div>

### $\text{Task e)}$

```bash
# Last timestep
Time = 100

Courant Number mean: 0.121869 max: 0.78421
smoothSolver:  Solving for Ux, Initial residual = 1.15982e-08, Final residual = 1.15982e-08, No Iterations 0
smoothSolver:  Solving for Uy, Initial residual = 1.14284e-08, Final residual = 1.14284e-08, No Iterations 0
DICPCG:  Solving for p, Initial residual = 1.25623e-06, Final residual = 8.66141e-07, No Iterations 1
time step continuity errors : sum local = 1.14492e-10, global = -1.62781e-19, cumulative = -1.16556e-17
DICPCG:  Solving for p, Initial residual = 8.95525e-07, Final residual = 8.95525e-07, No Iterations 0
time step continuity errors : sum local = 1.18287e-10, global = 5.42613e-20, cumulative = -1.16013e-17
ExecutionTime = 146.77 s  ClockTime = 147 s

End

```

<div style="display: flex; gap: 40px; text-align: center;">
  <div>
    <p><strong>Velocity Distrubution</strong></p>
    <img src="velocity_distrubution.png" width="1200">
  </div>

  <div>
    <p><strong>Stream function</strong></p>
    <img src="constant_psi.png" width="1200">
  </div>
</div>